# Naturalistic EMA prediction based on ACC - Dyskinesia use case

## 0) Imports

- document versions for reproducibility

In [ ]:
# import packages
import datetime as dt
import pandas as pd
import numpy as np
import os
import sys
import csv
import json
import importlib
from itertools import product, compress
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, variation

from dataclasses import dataclass, field


In [ ]:
print('Python sys', sys.version)
print('pandas', pd.__version__)
print('numpy', np.__version__)
# print('mne_bids', mne_bids.__version__)
# print('mne', mne.__version__)
# print('sci-py', scipy.__version__)
# print('sci-kit learn', sk.__version__)
# print('matplotlib', plt_version)

"""
Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.1.1
numpy 1.26.0

from 16.09

Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.3.2
numpy 2.3.3
"""

Import custom functions

In [ ]:
# from dbs_home repo
from dbs_home.load_raw.main_load_raw import loadSubject 
import dbs_home.utils.helpers as home_helpers
import dbs_home.utils.ema_utils as home_ema_utils
import dbs_home.plot_data.plot_compliance as plot_home_compl
import dbs_home.preprocessing.preparing_ema as home_ema_prep

from dbs_home.preprocessing import get_submovements
import dbs_home.preprocessing.acc_preprocessing as acc_prep
import dbs_home.preprocessing.submovement_processing as submove_proc
import dbs_home.load_raw.load_watch_raw as load_watch


In [ ]:
# from current repo
from utils import load_utils, load_data, prep_data
import utils.acc_features as acc_fts
import utils.ft_extract_class as ft_extr_class
import utils.feat_extraction as ft_extr
import utils.data_handling_ema_acc as data_handling

from plotting import plot_help



## 1) Load and check naturalistic data

In [ ]:
# set paths

feas_data_path = os.path.join(
    os.path.dirname(load_utils.get_onedrive_path()),
    'PROJECTS', 'home_feasibility'
)
feas_fig_path = os.path.join(
    load_utils.get_onedrive_path('figures'),
    'feasibility'
)



#### Load ACC data
- create SVM
- filter data within the dataclass

In [ ]:
# LID
sub_id = 'hm24'
ses_id = 'ses03'

FT_PARAMS_VERSION = 'v3'  # v3 updated lid version

In [ ]:
# import naturalistic data via dbs_home repo




### test days for hm24-ses01  # dyskinesia
# dev_day_selection = ['2025-07-17', '2025-07-18']
# dev_day_selection = [f'2025-07-{d}' for d in np.arange(17, 31)]
dev_day_selection = []

### test days for hm20-ses01  # tremor
# dev_day_selection = [
#     '2025-06-13', '2025-06-14',
#     '2025-06-15', '2025-06-16'
# ]

home_dat = loadSubject(
    sub=sub_id,
    ses=ses_id,
    incl_STEPS=False,
    incl_EPHYS=False,
    incl_EMA=True,
    incl_ACC=True,
    day_selection=dev_day_selection
)

Check available EMAs

In [ ]:
plot_home_compl.plot_EMA_completion_perSession(home_dat)

## 2) ACC processing incl. Feature extraction

#### 2a) Extract features from Acc-Windows aligned to EMAs

In [ ]:
importlib.reload(ft_extr_class)
importlib.reload(acc_fts)
importlib.reload(data_handling)
importlib.reload(ft_extr)


xy_dict = {}

# iter_settings = {
#     'nosm_allwindow':[False, True, True],
#     'sm_merged': [True, False, False],
#     'sm_single': [True, False, True]}

ft_settings = {
    'nosm_allwindow': {
        'EXTRACT_FT_FROM_SMs': False,
        'EXTRACT_FT_FULL_WIN': True,
        'ACC_FEATS_on_SINGLE_MOVES': True
    },
    'sm_merged': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': False
    },
    'sm_single': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': True
    }
}




for ft_type, ft_settings in ft_settings.items():

    xy_dict[ft_type] = ft_extr.get_features_per_session(
        home_dat=home_dat,
        sub_id=sub_id,
        ses_id=ses_id,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        LOAD_SAVE_FEATS = True,
        # define how features should be extracted
        STORE_SUBMOVES = False,
        # plotting settings
        SAVE_PLOT = False,
        SHOW_PLOT = False,
        **ft_settings
    )

In [ ]:
[f'{k}: {v.shape}, column-names: {list(v.keys())}' for k, v in xy_dict.items()]

Load data from different sessions (only EMA windows)

In [ ]:
importlib.reload(ft_extr)

FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single

xy_dict = {}


for ses_id in ['ses01', 'ses02', 'ses03']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
        # **ft_settings[FT_TYPE]
    )
    xy_dict[ses_id] = tempdf

#### 2b) feature extractions for full days

updated function to directly get feature dataframes, without home_dat requirement

In [ ]:
importlib.reload(acc_prep)
importlib.reload(load_watch)
importlib.reload(ft_extr)


In [ ]:
importlib.reload(ft_extr)

dfs_sessions = {}

sub_id = 'hm24'

for ses_id in ['ses01', 'ses02']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id, ses_id=ses_id,
        ft_set_sel = 'sm_single',
        ONLY_EMA_WINDOWS=False,
    )
    dfs_sessions[ses_id] = tempdf


Visualise and check submovement velocity and PCAs
- !! currently requires code parts that are internalised within ft_extr.get_features_per_session()

In [ ]:
from sklearn.decomposition import PCA


In [ ]:


# for i_sm, tempwin in enumerate(sm_win_data[:30]):

#     for att in ['pc1', 'pc2']:
#         plt.plot(tempwin.timestamps, getattr(tempwin, att), label=att,
#                 lw=3, alpha=.5,)

#     plt.plot(tempwin.timestamps, tempwin.svm, label='svm',)
#     plt.plot(tempwin.timestamps, tempwin.velocity.T,
#             label=[f'velo_{a}' for a in 'xyz'],
#             ls='--',)

#     plt.title(f'submovement # {i_sm}')
#     plt.legend()

#     plt.show()

In [ ]:


# plt.plot(tempwin.velocity.T)

# pca = PCA(n_components=2)
# projected = pca.fit_transform(tempwin.velocity.T)  # Shape: (N, 2)

# pc1 = projected[:, 0]  # Primary direction
# pc2 = projected[:, 1]  # Secondary direction

# plt.plot(pc1, alpha=.3, lw=5,)
# plt.plot(pc2, alpha=.3, lw=5,)

# plt.show()

In [ ]:
# plt.plot(np.abs(tempwin.velocity.T))

# pca = PCA(n_components=2)
# projected = pca.fit_transform(np.abs(tempwin.velocity.T))  # Shape: (N, 2)

# pc1 = projected[:, 0]  # Primary direction
# pc2 = projected[:, 1]  # Secondary direction

# plt.plot(pc1, alpha=.3, lw=5,)
# plt.plot(pc2, alpha=.3, lw=5,)

# plt.show()

visualise heartrate

In [ ]:
# importlib.reload(load_watch)

# hr_day_data = load_watch.get_source_heartrate_day(
#     sub=home_dat.sub, ses=home_dat.ses, date='2025-08-13',
# )

In [ ]:
# t1 = windat.acc_times[0]
# t2 = windat.acc_times[-1]

# hr_sel = np.logical_and(hr_day_data['timestamp'] > t1,
#                         hr_day_data['timestamp'] < t2)

# hr_win = hr_day_data[hr_sel].reset_index(drop=True)



# hr_win

In [ ]:
# fig, ax = plt.subplots(1, 1)

# ax2 = ax.twinx()

# ax.plot(windat.acc_times, windat.acc_svm)
# ax2.plot(hr_win['timestamp'], hr_win[' HeartRate'], color='orangered',)

# ax2.plot(hr_day_data['timestamp'], hr_day_data[' HeartRate'],
#          color='orangered',)


# plt.show()

## 3. Evaluate extracted Features

- Hssayeni et al, Scientific Reports 2021
    - strongest wrist-features: angular velocity, standard deviation, power of secondary frequency, power of 1–4 Hz band, and Shannon Entropy (r = 0.82  - r = 0.75)

- from svm: classic features
- include cross-corr between pc1 and pc2


In [ ]:
from plotting import plot_home_preds as plt_preds
import plotting.plot_ft_explor as plt_fts

In [ ]:
def col_strlist_to_mean(col_str):
    new_col = []

    for s in col_str:
       
        if isinstance(s, float) and not np.isnan(s):
            new_col.append(s)
            continue

        elif isinstance(s, float) and np.isnan(s):
            new_col.append(np.nan)
            continue

        
        if isinstance(s, str):
            if ', ' in s:
                s = s[1:-1].split(', ')
            elif ' ' in s:
                s = s[1:-1].split(' ')
            else:
                print(f'Unexpected string format: {s}')
            
        try:
            m = np.mean([float(val) for val in s if val != ''])
        except:
            print(f'Error processing string: {s}')
            
        new_col.append(m)

    return new_col

In [ ]:
xy_dict['ses01'].keys()

In [ ]:
xy_dict['ses01'].head()

In [ ]:
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
FT_PARAMS_DICT = ft_extr.get_feat_params(FT_PARAMS_VERSION)
feats_incl = [k for k,v in FT_PARAMS_DICT[0].items() if v]

noFeat_keys = ['timestamp'] + [FT_PARAMS_DICT[1].keys()]  # times + ema

In [ ]:
# FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
# ft_df = xy_dict[FT_TYPE]
# print(ft_df.keys())

# string_columns = ['zero_crossing_rate', 'axis_correlations']

# for col in ft_df.keys():
#     if any([s in col for s in string_columns]):
#         print(f'Converting column {col} from string to mean value.')
#         ft_df[col] = col_strlist_to_mean(ft_df[col])

# # ft_df['zero_crossing_rate'] = col_strlist_to_mean(ft_df['zero_crossing_rate'])
# # ft_df['axis_correlations'] = col_strlist_to_mean(ft_df['axis_correlations'])

In [ ]:
importlib.reload(plt_fts)

SES = 'ses01'
ft_df = xy_dict[SES]
save_plot = True
EMA_ref = 'LID'

for ft_name in ft_df.keys():
    print(f'\n{ft_name}')
    # if ft_name not in ft_df.keys():
    #     print(f'Feature {ft_name} not found in dataframe columns. Skipping.')
    #     continue
    if ft_name in noFeat_keys:
        print(f'Feature {ft_name} is not included in the feature set. Skipping.')
        continue

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0] = plt_fts.plot_ft_hist(
        df=ft_df,
        feat_name=ft_name,
        ax=axes[0],
        show=False,
    )

    axes[1] = plt_fts.scatter_feat_vs_EMA(
        df=ft_df,
        feat_name=ft_name,
        EMA_col_name=EMA_ref,
        ax=axes[1],
        show=False,
    )

    axes[2] = plt_fts.boxplot_feat_vs_EMA(
        df=ft_df,
        feat_name=ft_name,
        EMA_col_name=EMA_ref,
        ax=axes[2],
        show=False,
    )

    plt.tight_layout()

    if save_plot:
        lid_fts_figpath = os.path.join(
            load_utils.get_onedrive_path('emaval_fig'),
            'acc_lid_ft_explore', 
            f'sub24_{SES}_{FT_TYPE}_ft{FT_PARAMS_VERSION}'
        )
        if not os.path.exists(lid_fts_figpath):
            os.makedirs(lid_fts_figpath)
        plt.savefig(os.path.join(lid_fts_figpath,
                                 f'{ft_name}_{FT_PARAMS_VERSION}_explore.png'),
                    facecolor='w', dpi=300,)
        plt.close()

    else:
        plt.show()

### Predict EMAs

In [ ]:

from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold
from sklearn.metrics import r2_score, root_mean_squared_error

from sklearn.linear_model import ElasticNet, Lasso, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
# from sklearn.metrics import (
#     accuracy_score, r2_score,
#     roc_auc_score, balanced_accuracy_score
# )
# from scipy.stats import f as stats_f
# from sklearn.feature_selection import f_regression, r_regression

from utils import pred_nat_ema 

#### dev cross-validation n=1 concept

In [ ]:
# EXCL_HR = True
# ZSCORE_Y = False

# # # get array with shape n-windows, n-feats
# # X = np.array([l for l in list(FEAT_STORE.values())]).T
# # # get array with length n-windows
# # y = np.array(Y_STORE['LID']).astype(float).reshape(-1, 1)

# ft_set = list(xy_dict.keys())[2]
# print(f'\fFeature set chosen: {ft_set}')

# keys_excl_feats = [
#     'timestamp', 'LID', 'tremor', 'overall_move', 'move_hands', 'wellbeing'
# ]
# FTS_INCL = [k for k in list(xy_dict[ft_set])
#             if (k not in keys_excl_feats) and ('hr' not in k)]
# EMA_Y = 'LID'

# if MULTI_SESH:
#     ### MULTI-SESSION DATA
#     X = np.concat([X_ALL['ses01'], X_ALL['ses02']]).T
#     y = np.concat([y_ALL['ses01'], y_ALL['ses02']]).ravel()
# else:
#     ### SINGLE SESSION DATA
#     X = xy_dict[ft_set][FTS_INCL].values.copy()
#     y = xy_dict[ft_set][EMA_Y].values.astype(float)
#     X, y = pred_nat_ema.prep_X_y(X, y)  # delete nans, log, z-score


# # # WITHOUT HR BCS OF NANS
# # if EXCL_HR:
# #     hr_sel = ['hr_' in k for k in list(FEAT_STORE.keys())]
# #     X = X[~np.array(hr_sel), :]
# #     FTNAMES_KEPT = [n for n in list(FEAT_STORE.keys()) if 'hr_' not in n]
# # else:
# #     FTNAMES_KEPT = list(FEAT_STORE.keys())

# if ZSCORE_Y:
#     y = ((y - np.mean(y)) / np.std(y))
#     # y = (y - Z_y_mean) / Z_y_std


# # plt.plot(y)
# # plt.show()
# # print(X.shape, y.shape)

# # # WITHOUT HR BCS OF NANS
# # if EXCL_HR:
# #     hr_sel = ['hr_' in k for k in list(FEAT_STORE.keys())]
# #     X = X[:, ~np.array(hr_sel)]
# # print(X.shape, y.shape)

# # check where nans are, TODO during creation
# # list(compress(list(FEAT_STORE.keys()), np.any(np.isnan(X), axis=1)))

# # if X.shape[0] > X.shape[1]:
# #     nonnan_sel = ~np.any(np.isnan(X), axis=1)
# #     y = y[nonnan_sel]
# #     X = X[nonnan_sel, :]
# # else:
# #     nonnan_sel = ~np.any(np.isnan(X), axis=0)
# #     X = X[:, nonnan_sel]
# #     y = y[nonnan_sel]


# print(X.shape, y.shape, len(FTS_INCL))


#### compare different feature sets (full-window vs single-movements extracted)

In [ ]:
# for ft_set in list(xy_dict.keys()):

#       print(f'\n{"#" * 5}\nFeature set chosen: {ft_set}')

#       FTS_INCL = [k for k in list(xy_dict[ft_set])
#                   if (k not in keys_excl_feats) and ('hr' not in k)]
      
#       print(FTS_INCL)

In [ ]:
# importlib.reload(plot_home_preds)
# importlib.reload(pred_nat_ema)


# ### SELECT FEATURE SET for prediction

# ft_set = list(xy_dict.keys())[2]
# # print(f'\fFeature set chosen: {ft_set}')

# keys_excl_feats = [
#     'timestamp', 'LID', 'tremor', 'overall_move', 'move_hands', 'wellbeing'
# ]
# EMA_Y = 'LID'


# for ft_set in xy_dict.keys():

#       if ft_set != 'sm_single': continue

#       if ft_set.startswith('sm'): EXCL_ZEROS = True
#       else: EXCL_ZEROS = False

#       print(f'\n{"#" * 50}\nFeature set chosen: {ft_set}')

#       FTS_INCL = [k for k in list(xy_dict[ft_set])
#                   if (k not in keys_excl_feats) and ('hr' not in k)]

#       X = xy_dict[ft_set][FTS_INCL].values.copy()
#       y = xy_dict[ft_set][EMA_Y].values.copy().astype(float)
#       # get day array for crossval leave day out
#       days = xy_dict[ft_set]['timestamp']  # get timestamps per sample
#       days = [d.split(' ')[0] for d in days]  # only take day string per timestamp
#       day_codes = [np.where(np.unique(days) == d)[0][0] for d in days]  # convert strings into codes per day
      
#       # TODO: include day in prep X y 
#       X, y, excluded_bool = prep_X_y(
#             X, y, REMOVE_ZERO_SUBMOVES=EXCL_ZEROS, verbose=False,
#             RETURN_EXCLUDED_NAN_BOOL=True,
#       )  # delete nans, log, z-score
#       if sum(excluded_bool) > 0:
#             day_codes = np.array(day_codes)[~excluded_bool]
#             assert len(day_codes) == X.shape[0], 'incorrect day code length vs X'

#       ### predict
#       PERMS = False
#       N_PERMS = 1000
#       TO_PLOT = True

#       CLS_METHOD = 'lasso'
#       n_fold_cv = 3

#       if not PERMS:
#             y_pred_total = pred_nat_ema.classif_home_ema(
#                   X=X, y=y, PERMS=PERMS, N_PERMS=N_PERMS, n_fold_cv=n_fold_cv,
#                   CLS_METHOD=CLS_METHOD, ZSCORE_Y=ZSCORE_Y, verbose=False,
#                   day_codes=day_codes, leave_day_out_cv=True,
#             )

#       else:
#             print('\n\n ############\nADD PERMUTATION FLOW')


#       #### PLOTTING
#       if TO_PLOT:
#             if MULTI_SESH:
#                   FIGNAME = f'LidPred_{sub_id}_ses0102_{CLS_METHOD}_scatter_n{len(y)}'
#             else:
#                   FIGNAME = f'LidPred_{sub_id}_{ses_id}_{CLS_METHOD}_scatter_n{len(y)}'
#             if EXCL_HR: FIGNAME += '_exclHR'
#             else: FIGNAME += '_inclHR'

#             plot_home_preds.scatter_preds(y=y, y_pred_total=y_pred_total,
#                                           ZSCORE_Y=ZSCORE_Y, show=TO_PLOT,)



      

## Xa) Use-case LID-classification:

- only EMA windows
- cv-folds: 5-folds of EMA-windows from all sessions

In [ ]:
from utils.pred_utils import fit_cv_regr

In [ ]:
FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
SES = 'ses01'
# ema_df = xy_dict[FT_TYPE]
pred_df = xy_dict[SES]
print(pred_df.keys())


# string_columns = ['zero_crossing_rate', 'axis_correlations']

# for col in ema_df.keys():
#     if any([s in col for s in string_columns]):
#         print(f'Converting column {col} from string to mean value.')
#         ema_df[col] = col_strlist_to_mean(ema_df[col])

In [ ]:
importlib.reload(ft_extr)
importlib.reload(data_handling)

### load feature data

sub_id = 'hm24'
EMA_Y = 'LID'

EXCL_HR = True

# CLS_METHOD = 'elasticnet'


# ema_dfs = []

# # load training data: only windows with corresponding EMA
# for ses_id in ['ses01', 'ses02', 'ses03'][:-2]:
#     ema_dfs.append(
#         ft_extr.get_feat_df_for_pred(
#             sub_id=sub_id,
#             ses_id=ses_id,
#             ft_set_sel=feat_selection,
#             ONLY_EMA_WINDOWS=True,
#         ).reset_index(drop=True)
#     )
#     print(f'{ses_id} done, added shape {ema_dfs[-1].shape}')

# ema_df = pd.concat(ema_dfs, ignore_index=True)

In [ ]:
keys_excl_feats = [
    'timestamp', 'LID', 'tremor', 'overall_move', 'move_hands', 'wellbeing'
]

### select features
if EXCL_HR:
    FTS_INCL = [
        k for k in list(pred_df.keys())
        if (k not in keys_excl_feats) and ('hr' not in k)
    ]
else:
    FTS_INCL = [
        k for k in list(pred_df.keys()) if k not in keys_excl_feats
    ]
print(f'model trained on features: {FTS_INCL}')

### define training X, y
X_all = pred_df[FTS_INCL].values.copy()
y_all = pred_df[EMA_Y].values.copy().astype(float)
print('pre prep', X_all.shape, y_all.shape)
nan_rows = np.any(np.isnan(X_all), axis=1)
X_all = X_all[~nan_rows]
y_all = y_all[~nan_rows]


# outputs = pred_nat_ema.prep_X_y(
#     X_all, y_all,
#     REMOVE_ZERO_SUBMOVES=False,
#     RETURN_ZSCORE_PARAMS=False,
#     verbose=True,
# )  # delete nans, log, z-score
# # X_all, y_all, z_param_list = outputs
# X_all, y_all = outputs  # no scaling since this happens withi the pipe

print('post prep', X_all.shape, y_all.shape)
# check for nans
print('any nans in X_all?', np.any(np.isnan(X_all)))
print('any nans in y_all?', np.any(np.isnan(y_all)))

### transform y
# y_all[y_all == 1] = 1
# y_all[(y_all > 1) & (y_all <= 4)] = 2
# y_all[y_all > 4] = 3

In [ ]:
# pipe includes scaling based on training data, and regression model
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
    # ('reg', LogisticRegression(multi_class='multinomial',)),
])


In [ ]:
y_pred_all = np.zeros_like(y_all)

n_folds = 5
cv_folds = KFold(n_splits=n_folds).split(X_all)

for i_fold, (train_idx, test_idx) in enumerate(cv_folds):
    print(f'test indices for fold {i_fold + 1}: {test_idx}')
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    # train regression model
    pipe.fit(X_train, y_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred = pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred = np.clip(np.round(y_pred), y_train.min(), y_train.max()).astype(int)
    y_pred_all[test_idx] = y_pred  # add predictions to total array in correct indices
    
    print(f'finished fold {i_fold + 1} / {n_folds}')

In [ ]:
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import mean_absolute_error, cohen_kappa_score


quick evaluation, check sign with permutations

In [ ]:
rho, p = spearmanr(y_all, y_pred_all)
print(f"Spearman rho={rho.round(2)}, p={p.round(5)}")

tau, p_tau = kendalltau(y_all, y_pred_all)
print(f"Kendall tau={tau.round(2)}, p={p_tau.round(5)}")  

mae = mean_absolute_error(y_all, y_pred_all)
kappa = cohen_kappa_score(y_all, y_pred_all, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_all, y_pred_all)
rmse = root_mean_squared_error(y_all, y_pred_all)

print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

y_jitter = np.random.normal(0, 0.1, size=y_all.shape)  # add jitter for better visibility
ax.scatter(y_all + y_jitter, y_pred_all, alpha=0.5)
ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

## Xb) Use-case LID-classification:

- Train-split: EMA-windows ses01, Test-split: all windows, all sessions

In [ ]:
importlib.reload(ft_extr)
importlib.reload(data_handling)

### load feature data

sub_id = 'hm24'

feat_selection = 'sm_single'
# feat_selection = 'nosm_allwindow'

# load training data: only windows with corresponding EMA
FT_DF_TRAIN = ft_extr.get_feat_df_for_pred(
    sub_id=sub_id,
    ses_id='ses01',
    ft_set_sel=feat_selection,
    ONLY_EMA_WINDOWS=True,
).reset_index(drop=True)

# load test data: windows from whole sessions, without EMA
test_dfs = {}

for ses_id in ['ses01', 'ses02', 'ses03']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id, ses_id=ses_id,
        ft_set_sel=feat_selection,
        ONLY_EMA_WINDOWS=False,
    )
    test_dfs[ses_id] = tempdf

In [ ]:
FT_DF_TRAIN.keys()

In [ ]:
importlib.reload(pred_nat_ema)

EXCL_HR = False
keys_excl_feats = [
    'timestamp', 'LID', 'tremor', 'overall_move', 'move_hands', 'wellbeing'
]
EMA_Y = 'LID'

CLS_METHOD = 'lasso'

CLS_LIB = {
    'linreg': LinearRegression(),
    'lda': LDA(),  # only applicable on categorical or binary values
    'lasso': Lasso(alpha=0.3)
}

### select features
if EXCL_HR:
    FTS_INCL_train = [
        k for k in list(FT_DF_TRAIN.keys())
        if (k not in keys_excl_feats) and ('hr' not in k)
    ]
else:
    FTS_INCL_train = [
        k for k in list(FT_DF_TRAIN.keys()) if k not in keys_excl_feats
    ]
print(f'model trained on features: {FTS_INCL_train}')

### define training X, y
X_ema_train = FT_DF_TRAIN[FTS_INCL_train].values.copy()
y_ema_train = FT_DF_TRAIN[EMA_Y].values.copy().astype(float)

X_ema_train, y_ema_train, z_param_list = pred_nat_ema.prep_X_y(
    X_ema_train, y_ema_train,
    REMOVE_ZERO_SUBMOVES=False,
    RETURN_ZSCORE_PARAMS=True,
    verbose=False,
)  # delete nans, log, z-score

# train model
clf = CLS_LIB[CLS_METHOD]
clf.fit(X_ema_train, y_ema_train)
print(f'train data size: {X_ema_train.shape}')

### store test predictions
test_pred_dict, test_stamps_dict = {}, {}

for test_ses in test_dfs.keys():
    print(f'\n\ttest predict {test_ses}')

    ### define test X
    FT_DF_TEST = test_dfs[test_ses]
    if EXCL_HR:
        FTS_INCL_test = [
            k for k in list(FT_DF_TEST.keys())
            if (k not in keys_excl_feats) and ('hr' not in k)
        ]
    else:
        FTS_INCL_test = [
            k for k in list(FT_DF_TEST.keys())
            if k not in keys_excl_feats
        ]
    # compare features
    assert FTS_INCL_test == FTS_INCL_train, 'different feature sets'
    # select included features
    X_all_test = FT_DF_TEST[FTS_INCL_test].values.astype(float)

    # get corresponding timestamps
    test_stamps = [
        dt.datetime.strptime(t[:19], "%Y-%m-%d %H:%M:%S")
        for t in FT_DF_TEST.index.values
    ]
    
    X_all_test, _ = pred_nat_ema.prep_X_y(
        X_all_test, y=[],
        REMOVE_ZERO_SUBMOVES=True,
        USE_GIVEN_ZSCORE_PARAM=True,
        ZSCORE_PARAMS=z_param_list, 
        verbose=False,
    )  # delete nans, log, z-score

    assert X_ema_train.shape[1] == X_all_test.shape[1], 'different array-ft-lengths'

    ### create predictions
    test_pred_dict[test_ses] = clf.predict(X_all_test)
    test_stamps_dict[test_ses] = test_stamps
    print(f'\ttest data ({test_ses}) size: {X_all_test.shape}')
    

In [ ]:
importlib.reload(data_handling)

### b) TODO: Evaluate predictions
- load all EMAs, find nearest predictions, calculate metrics


### c) Visualize predictions


1) Plot overall session comparison

In [ ]:
from scipy.stats import mannwhitneyu

In [ ]:
SES_COLORS = {'ses01': 'orange', 'ses02': 'violet', 'ses03': 'darkcyan'}


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))

ax.boxplot([v for v in list(test_pred_dict.values())])

ax.set_ylabel('Dyskinesia prediction\n(EMA scale)')
ax.set_xticklabels(list(test_pred_dict.keys()))

plt.show()

for comp_ses in ['ses01', 'ses02']:
    S, p = mannwhitneyu(test_pred_dict[comp_ses], test_pred_dict['ses03'])

    print(f'optimized-DBS vs {comp_ses}: stat = {S.round(2)}, p = {p.round(4)}')

S, p = mannwhitneyu(test_pred_dict['ses01'], test_pred_dict['ses02'])

print(f'pre-op vs not-optimal-DBS: stat = {S.round(2)}, p = {p.round(4)}')

2) Plot predictions versus l-dopa intake moments


In [ ]:
### subplots per ses

FS=14

fig, axes = plt.subplots(len(test_pred_dict), 1,
                         figsize=(3*len(test_pred_dict), 8),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )


   bp = axes[i_ses].boxplot(box_distance_groups, patch_artist=True,)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

   axes[i_ses].set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
   axes[i_ses].set_xticklabels(dist_labels,)
   axes[i_ses].set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
   axes[i_ses].set_ylim(-2, 9)
   axes[i_ses].set_yticks([1, 3, 5, 7, 9])
   axes[i_ses].set_yticklabels([1, 3, 5, 7, 9])
   axes[i_ses].tick_params(size=FS, labelsize=FS,)

   axes[i_ses].set_title(ses_id, fontsize=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
# plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
#             facecolor='w', dpi=300,)

plt.close()

In [ ]:
### all sessions next to each other

FS=14
pos_adj = [0., .25, .5]

fig, ax = plt.subplots(1, 1, figsize=(9, 3),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )

   bp = ax.boxplot(
      box_distance_groups, patch_artist=True, label=ses_id,
      positions=np.arange(len(box_distance_groups)) + pos_adj[i_ses],
      widths=.25,
   )

   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
ax.set_xticks(np.arange(len(dist_labels))+pos_adj[1],)
ax.set_xticklabels(dist_labels,)
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.set_ylim(-3, 9)
ax.set_yticks([-3, -1, 1, 3, 5, 7, 9])
ax.set_yticklabels([-3, -1, 1, 3, 5, 7, 9])
ax.tick_params(size=FS, labelsize=FS,)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()

3) Plot predictions versus daytime

In [ ]:
importlib.reload(plot_home_preds)

Full-day curve

In [ ]:
### daily plot
FS=14

fig, axes = plt.subplots(len(test_pred_dict), 1,
                         figsize=(3*len(test_pred_dict), 8),
                         sharex='col',)

for i_ses, ses_id in enumerate(test_pred_dict.keys()):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]
   (
        daily_minutes, daily_mean, daily_std
    ) = data_handling.get_ft_daily_mean(test_pred, test_stamps,)
   
   plot_home_preds.plot_daily_ft_mean(
        daily_minutes, daily_mean, daily_std, 'LID prediction',
        use_ax=axes[i_ses], FS=FS, plot_color=SES_COLORS[ses_id]
   )

   # edit subplot
   axes[i_ses].set_title(f'Session: {ses_id}', fontsize=FS,)
   axes[i_ses].set_ylabel('LID prediction\n(EMA scale)', fontsize=FS,)
   axes[i_ses].set_ylim(1, 9)
   axes[i_ses].set_yticks([1, 3, 5, 7, 9])
   axes[i_ses].tick_params(labelsize=FS, size=FS,)


axes[-1].set_xlabel('Time at Day (hours)', fontsize=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_daytime'
# plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
#             facecolor='w', dpi=300,)

plt.close()

# plt.show()

Daytime blocks

In [ ]:
DAY_HOUR_BLOCKS = {'morning': [8, 12],
                   'afternoon1': [12, 15],
                   'afternoon2': [15, 18], 
                   'evening': [18, 22]}

box_lists = {k: {k2: [] for k2 in DAY_HOUR_BLOCKS}
             for k in test_pred_dict}


for ses_id in test_pred_dict.keys():

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]
   stamp_hrs = [data_handling.get_dayminutes(t)/60 for t in test_stamps]
   
   for t_hr, v in zip(stamp_hrs, test_pred):
      if t_hr < DAY_HOUR_BLOCKS['morning'][1]:
         box_lists[ses_id]['morning'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon1'][1]:
         box_lists[ses_id]['afternoon1'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon2'][1]:
         box_lists[ses_id]['afternoon2'].append(v)
      else:
         box_lists[ses_id]['evening'].append(v)

boxparams = {
   'ses01': {'widths': .2, 'positions': np.arange(.0, len(DAY_HOUR_BLOCKS)),},
   'ses02': {'widths': .2, 'positions': np.arange(.2, len(DAY_HOUR_BLOCKS)),},
   'ses03': {'widths': .2, 'positions': np.arange(.4, len(DAY_HOUR_BLOCKS)),}
}
FS=14

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for i_ses, ses_id in enumerate(box_lists):
   ses_lists = [l for l in box_lists[ses_id].values()]
   bp = ax.boxplot(ses_lists, **boxparams[ses_id],
                   patch_artist=True, label=ses_id)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))
ax.set_xticks(list(boxparams.values())[1]['positions'])
ax.set_xticklabels(list(DAY_HOUR_BLOCKS.keys()))
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.tick_params(labelsize=FS, size=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_daytimeBlocks'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()

### Significance testing Perm test

In [ ]:
# N_PERMS = 1000

# # TEST_SEL = daycode_arr > 20  # takes circa .33

# # X_train = X_arr[~TEST_SEL, :]
# # y_train = y_arr[~TEST_SEL]

# # # test cohort
# # X_test = X_arr[TEST_SEL, :]
# # y_true = y_arr[TEST_SEL]

# model = LDA()

# # Run prediction
# perm_mets ={'F': [], 'R': []}

# y_true = y_arr.astype(int)

# np.random.seed(27)

# for i_perm in np.arange(N_PERMS):

#     y_perm_pred = np.array([np.nan] * len(y_arr))

#     for day in np.unique(daycode_arr):

#         TEST_SEL = daycode_arr == day
#         X_train = X_arr[~TEST_SEL, :]
#         y_train = y_arr.astype(int)[~TEST_SEL]
#         np.random.shuffle(y_train)   

#         # test cohort
#         X_test = X_arr[TEST_SEL, :]

#         model = models[CLSF]
#         model.fit(X_train, y_train)

#         y_perm_pred[TEST_SEL] = model.predict(X_test)

#     # y_train = y_arr[~TEST_SEL]
#     # np.random.shuffle(y_train)   
#     # model.fit(X_train, y_train)

#     # y_perm_pred = model.predict(X_test)
#     # # np.random.shuffle(y_perm_pred)
#     y_perm_pred = np.array([np.round(v) for v in y_perm_pred])

#     F, f_pvalue = get_f_stat(y_pred=y_perm_pred, y_true=y_true,
#                              n_feats=X_test.shape[1])
#     prs_stat, _ = pearsonr(y_perm_pred, y_true)
#     perm_mets['F'].append(F)
#     perm_mets['R'].append(prs_stat)




In [ ]:
# # if MULTI_SESH:
# #     FIGNAME = f'LidPred_{sub_id}_SESS0102_{CLS_METHOD}_PermSign_n{N_PERMS}'
# # else:
# #     FIGNAME = f'LidPred_{sub_id}_{ses_id}_{CLS_METHOD}_PermSign_n{N_PERMS}'


# fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# # pred_mets = {'F': pred_F, 'R': pred_corrcoef}

# for i_ax, metr in enumerate(['F', 'R']):

#     axes[i_ax].hist(perm_results[metr], color='gray', alpha=.5,)
#     axes[i_ax].axvline(np.percentile(perm_results[metr], 97.5),
#                        color='orange', alpha=.8, lw=3,
#                        label=f'alpha=0.025\n(n-perm: {N_PERMS})',)
    
#     axes[i_ax].axvline(perm_true_results[metr],
#                        color='purple', alpha=.5, lw=1,
#                        label='prediction',)
    
#     p_calc = sum(perm_true_results[metr] < perm_results[metr]) / len(perm_results[metr])
#     print(f'metric {metr}: p = {np.round(p_calc, 3)}')

#     axes[i_ax].set_xlabel(f'{metr} score', size=14,)

#     axes[i_ax].set_ylabel('count (n)', size=14)

#     axes[i_ax].tick_params(axis='both', size=14, labelsize=14,)
#     axes[i_ax].spines[['right', 'top']].set_visible(False)

# axes[1].legend(frameon=False, fontsize=14,
#                bbox_to_anchor=(.95, .5), loc='center left')

# plt.tight_layout()

# # plt.savefig(os.path.join(load_utils.get_onedrive_path('figures'),
# #              'proof_kin_pred', FIGNAME),
# #              dpi=300, facecolor='w',)

# plt.show()

## Use case Tremor clustering

### a) Data loading

In [ ]:
importlib.reload(ft_extr)
importlib.reload(data_handling)

### load feature data

sub_id = 'hm20' # 'hm22'

feat_selection = 'sm_single'
# feat_selection = 'nosm_allwindow'

# load training data: only windows with corresponding EMA
FT_DF_CLUST = ft_extr.get_feat_df_for_pred(
    sub_id=sub_id,
    ses_id='ses01',
    ft_set_sel=feat_selection,
    ONLY_EMA_WINDOWS=True,
)


### b) Feature selection and clustering